# Graph model results

Reads artifacts under `saved/graph_models/` written by `03_graph_models.ipynb`. Does **not** retrain.

Compares:

- baseline GraphSAGE / GAT (`gnn_results_*.csv`, `gnn_vs_tabular_*.csv`)
- SIGN GraphSAGE v2 (`gnn_v2_results_*.csv`, `gnn_v2_metadata_*.json`)
- compact RF / LightGBM / XGBoost from the tabular notebooks

Kernel: `ai` (pandas). Training is `03_graph_models.ipynb`.


In [ ]:
import json
import warnings
from pathlib import Path

import pandas as pd

warnings.filterwarnings("ignore")
pd.options.display.precision = 4

try:
    import google.colab
    IS_COLAB = True
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    IS_COLAB = False

ROOT = Path("/content/drive/MyDrive/minor-thesis") if IS_COLAB else Path.cwd()
DATASET_PATH = ROOT / "dataset"
SAVED_PATH = ROOT / "saved"
RESULTS_DIR = SAVED_PATH / "graph_models"
print(f"Running on {'Google Colab' if IS_COLAB else 'Local'}")
print(f"Results: {RESULTS_DIR}")
print(f"Exists: {RESULTS_DIR.exists()}")


## Baseline GraphSAGE vs tabular models


In [6]:
runs = sorted((SAVED_PATH / "graph_models").glob("gnn_vs_tabular_*.csv"))
if runs:
    latest = runs[-1]
    print(f"Loaded {latest.name}")
    cmp = pd.read_csv(latest)
    display(cmp[["model", "roc_auc", "pr_auc", "f1", "recall"]].sort_values("roc_auc", ascending=False))
else:
    print("No gnn_vs_tabular_*.csv yet.")


Loaded gnn_vs_tabular_20260820_161837.csv


,model,roc_auc,pr_auc,f1,recall
2,RF - Remove V,0.9133,0.5445,0.4223,0.2835
3,RF - Remove id + V,0.9114,0.5450,0.4307,0.2904
6,LightGBM - Remove id + V,0.9108,0.5020,0.3250,0.7505
5,LightGBM - Remove V,0.9098,0.5174,0.3254,0.7490
9,XGBoost - Remove id + V,0.9078,0.4986,0.3363,0.7340
8,XGBoost - Remove V,0.9076,0.5137,0.3348,0.7335
4,LightGBM - Baseline,0.9057,0.5203,0.3355,0.7274
1,RF - Baseline,0.9056,0.5279,0.4587,0.3206
7,XGBoost - Baseline,0.9017,0.5170,0.3551,0.7124
0,GraphSAGE,0.7667,0.1443,0.1180,0.7734


In [ ]:
runs = sorted(RESULTS_DIR.glob("gnn_vs_tabular_*.csv"))
if not runs:
    print("No saved comparison yet — run the training cell first.")
else:
    latest = runs[-1]
    cmp = pd.read_csv(latest)
    print(f"Loaded {latest.name}")
    display(cmp[["model", "roc_auc", "pr_auc", "f1", "recall", "precision"]].sort_values("roc_auc", ascending=False))

meta_files = sorted(RESULTS_DIR.glob("gnn_metadata_*.json"))
if meta_files:
    with open(meta_files[-1], encoding="utf-8") as f:
        meta = json.load(f)
    print("Artifacts")
    print(f"  timestamp : {meta['timestamp']}")
    print(f"  device    : {meta['device']}")
    print(f"  features  : {meta['num_features']}")
    print(f"  nodes     : {meta['num_nodes']:,}")

## SIGN GraphSAGE (v2) vs baseline


In [ ]:
v2_runs = sorted(RESULTS_DIR.glob("gnn_v2_results_*.csv"))
base_runs = sorted(RESULTS_DIR.glob("gnn_results_*.csv"))
meta_v2 = sorted(RESULTS_DIR.glob("gnn_v2_metadata_*.json"))

rows = []
if base_runs:
    base = pd.read_csv(base_runs[-1])
    base["source_file"] = base_runs[-1].name
    rows.append(base)
    print(f"Baseline: {base_runs[-1].name}")
if v2_runs:
    v2 = pd.read_csv(v2_runs[-1])
    v2["source_file"] = v2_runs[-1].name
    rows.append(v2)
    print(f"v2: {v2_runs[-1].name}")

if rows:
    gnn = pd.concat(rows, ignore_index=True)
    cols = [c for c in ["model", "threshold", "roc_auc", "pr_auc", "f1", "mcc", "recall", "precision", "tn", "fp", "fn", "tp", "best_epoch"] if c in gnn.columns]
    display(gnn[cols].sort_values("roc_auc", ascending=False))
else:
    print("No gnn_results_*.csv or gnn_v2_results_*.csv yet. Run 03_graph_models.ipynb first.")

if meta_v2:
    with open(meta_v2[-1], encoding="utf-8") as f:
        meta = json.load(f)
    print(f"\nv2 metadata: {meta_v2[-1].name}")
    print(f"  variant     : {meta.get('variant')}")
    print(f"  best_epoch  : {meta.get('best_epoch')}")
    print(f"  pos_weight  : {meta.get('pos_weight')}")
    print(f"  threshold   : {meta.get('threshold_mcc')}")
    prev = meta.get("previous") or {}
    mcc = meta.get("metrics_mcc_threshold") or {}
    if prev and mcc:
        print(f"  ROC-AUC     : {mcc.get('roc_auc'):.4f}  (baseline {prev.get('roc_auc'):.4f})")
        print(f"  TP / TN / FP: {mcc.get('tp')} / {mcc.get('tn')} / {mcc.get('fp')}  (baseline {prev.get('tp')} / {prev.get('tn')} / {prev.get('fp')})")


## Saved files

Under `saved/graph_models/`:

- `graphsage_YYYYMMDD_HHMMSS.pt` / `gat_YYYYMMDD_HHMMSS.pt` — baseline weights
- `gnn_results_*.parquet` / `.csv` — GraphSAGE and GAT holdout metrics
- `gnn_val_predictions_*.parquet` — baseline validation probabilities
- `gnn_vs_tabular_*.parquet` / `.csv` — comparison with RF / LightGBM / XGBoost
- `gnn_metadata_*.json` — baseline features, split, history
- `graphsage_v2_YYYYMMDD_HHMMSS.pt` — SIGN GraphSAGE weights
- `gnn_v2_results_*.csv` — v2 holdout metrics at 0.5 and MCC threshold
- `gnn_v2_metadata_*.json` — v2 edge keys, history, confusion counts

Also written during the importance pass:

- `saved/graph_edge_schema.json`
- `saved/model_column_importance.parquet`
- `saved/lgb_column_importance.parquet`
- `saved/ablation_group_importance.parquet`
